# Phase 10 — Data Quality & Schema Enforcement Experiments Notebook

This is the **worked SOLUTION notebook** for Phase 10.

Run it **top-to-bottom**. The repeated workflow is:

```text
state the contract
    ↓
state the grain
    ↓
classify the rule
    ↓
predict invalid rows
    ↓
implement ONE validation concern
    ↓
preserve rejection evidence
    ↓
split accepted/rejected
    ↓
reconcile counts
    ↓
review Spark execution
```

Core question:

> **Can this validation layer protect downstream transformations without silently deleting bad data or scattering rules throughout the pipeline?**

Important:

- Examples target **PySpark 4.2.0**.
- Python strings are single-quoted.
- Important code includes inline teaching comments.
- Validation rules are explicit rather than hidden inside a framework.
- Rejected rows preserve diagnostic evidence.
- The notebook does **not** perform the formal Phase 10 mastery gate, update `ROADMAP.md`, mark Phase 10 complete, or enter Phase 11.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Data-Quality Exercise Protocol](#exercise-protocol)
- [Experiment 1 — Structural Schema Contract](#experiment-1)
- [Experiment 2 — Required Fields](#experiment-2)
- [Experiment 3 — Domain, Range, and Business Rules](#experiment-3)
- [Experiment 4 — Primary-Key Uniqueness](#experiment-4)
- [Experiment 5 — Composite-Key Validation](#experiment-5)
- [Experiment 6 — Exact Duplicates vs. Duplicate Business Keys](#experiment-6)
- [Experiment 7 — Referential Integrity](#experiment-7)
- [Experiment 8 — Multiple Rejection Reasons](#experiment-8)
- [Experiment 9 — Accepted vs. Rejected Records](#experiment-9)
- [Experiment 10 — Quarantine Design](#experiment-10)
- [Experiment 11 — Validation Metrics and Reconciliation](#experiment-11)
- [Experiment 12 — Distributed Cost of Validation](#experiment-12)
- [Applied Phase 10 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Declared grains:

```text
orders_df
= one row per order_id

customers_df
= one row per customer_id

inventory_df
= one row per snapshot_date + store_id + product_id
```

The data intentionally contains quality defects so the validation layer has something real to detect.

[Back to Table of Contents](#toc)


In [ ]:
from datetime import date
from decimal import Decimal

from pyspark.sql import Column
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.types import DecimalType
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType


spark = (
    SparkSession.builder
    .appName('phase_10_data_quality_experiments')
    .master('local[4]')
    # Keep the teaching workload small and predictable.
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
# Incoming schemas allow NULL values so invalid rows can be diagnosed
# instead of failing before the quality layer sees them.
ORDERS_SCHEMA = StructType(
    [
        StructField('order_id', StringType(), nullable=True),
        StructField('customer_id', StringType(), nullable=True),
        StructField('order_date', DateType(), nullable=True),
        StructField('order_status', StringType(), nullable=True),
        StructField('net_sales', DecimalType(12, 2), nullable=True),
    ]
)

CUSTOMERS_SCHEMA = StructType(
    [
        StructField('customer_id', StringType(), nullable=True),
        StructField('customer_name', StringType(), nullable=True),
        StructField('province', StringType(), nullable=True),
        StructField('customer_segment', StringType(), nullable=True),
    ]
)

INVENTORY_SCHEMA = StructType(
    [
        StructField('snapshot_date', DateType(), nullable=True),
        StructField('store_id', StringType(), nullable=True),
        StructField('product_id', StringType(), nullable=True),
        StructField('quantity_on_hand', IntegerType(), nullable=True),
    ]
)

ORDERS_ROWS = [
    ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('125.00')),
    (None, 'C002', date(2026, 9, 1), 'COMPLETED', Decimal('80.00')),
    ('O003', 'C001', date(2026, 9, 2), 'UNKNOWN', Decimal('45.00')),
    ('O004', 'C003', date(2026, 9, 2), 'COMPLETED', Decimal('-20.00')),
    ('O005', 'C999', date(2026, 9, 3), 'COMPLETED', Decimal('30.00')),
    ('O006', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('10.00')),
    ('O006', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('15.00')),
    ('O007', 'C004', date(2026, 9, 4), 'PENDING', Decimal('20.00')),
    ('O007', 'C004', date(2026, 9, 4), 'PENDING', Decimal('20.00')),
    ('O008', 'C003', date(2026, 9, 4), 'CANCELLED', Decimal('12.00')),
    ('O009', 'C002', date(2026, 9, 5), 'CANCELLED', Decimal('0.00')),
]

CUSTOMERS_ROWS = [
    ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ('C002', 'Ben Tremblay', 'QC', 'CONSUMER'),
    ('C003', 'Carla Singh', 'BC', 'BUSINESS'),
    ('C004', 'Diego Martin', 'ON', 'CONSUMER'),
]

INVENTORY_ROWS = [
    (date(2026, 9, 5), 'S001', 'P001', 10),
    (date(2026, 9, 5), 'S001', 'P002', 7),
    (date(2026, 9, 5), 'S002', 'P001', 5),
    (date(2026, 9, 5), 'S002', 'P001', 8),
    (date(2026, 9, 5), 'S003', 'P003', -1),
]

orders_df = spark.createDataFrame(ORDERS_ROWS, schema=ORDERS_SCHEMA)
customers_df = spark.createDataFrame(CUSTOMERS_ROWS, schema=CUSTOMERS_SCHEMA)
inventory_df = spark.createDataFrame(INVENTORY_ROWS, schema=INVENTORY_SCHEMA)

orders_df.orderBy(F.col('order_id').asc_nulls_first()).show(truncate=False)
customers_df.orderBy('customer_id').show(truncate=False)
inventory_df.orderBy('snapshot_date', 'store_id', 'product_id').show(truncate=False)


<a id="exercise-protocol"></a>
# Data-Quality Exercise Protocol

For each validation concern:

```text
1. State the contract.
2. State the grain.
3. Classify the rule:
   row-level / dataset-level / cross-dataset.
4. Predict which rows are invalid.
5. Predict whether Spark needs:
   narrow expressions / shuffle / join / action.
6. Implement the rule.
7. Preserve diagnostic reasons.
8. Verify grain and classification.
9. Reconcile counts.
10. Explain the downstream defect prevented.
```

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Structural Schema Contract

Contract:

```text
orders_df must contain the expected columns with the expected Spark data types.
```

This is a **structural** check. It is not a row-level content rule.

Prediction:

```text
correct orders_df
→ pass

missing net_sales
→ fail structural contract

net_sales as string
→ fail structural contract
```

[Back to Table of Contents](#toc)


In [ ]:
def validate_schema(
    df: DataFrame,
    expected_schema: StructType,
    label: str,
    allow_extra_columns: bool = False,
) -> None:
    '''Validate required columns and Spark data types.'''

    # Schema metadata is available on the driver, so this does not require
    # scanning the dataset with a Spark job.
    expected_fields = {
        field.name: field.dataType
        for field in expected_schema.fields
    }
    actual_fields = {
        field.name: field.dataType
        for field in df.schema.fields
    }

    missing_columns = set(expected_fields) - set(actual_fields)
    unexpected_columns = set(actual_fields) - set(expected_fields)

    type_mismatches = {
        column_name: (
            expected_fields[column_name],
            actual_fields[column_name],
        )
        for column_name in expected_fields.keys() & actual_fields.keys()
        if expected_fields[column_name] != actual_fields[column_name]
    }

    if missing_columns:
        raise ValueError(
            f'{label} is missing required columns: {sorted(missing_columns)}'
        )

    if type_mismatches:
        raise ValueError(
            f'{label} contains type mismatches: {type_mismatches}'
        )

    if unexpected_columns and not allow_extra_columns:
        raise ValueError(
            f'{label} contains unexpected columns: {sorted(unexpected_columns)}'
        )


# Correct structure passes.
validate_schema(
    orders_df,
    expected_schema=ORDERS_SCHEMA,
    label='orders_df',
)

print('Correct orders schema passed.')


In [ ]:
# Missing net_sales should fail because the dataset contract is incomplete.
orders_missing_column_df = orders_df.drop('net_sales')

try:
    validate_schema(
        orders_missing_column_df,
        expected_schema=ORDERS_SCHEMA,
        label='orders_missing_column_df',
    )
except ValueError as error:
    print(error)


In [ ]:
# Cast net_sales to string to create a structural type mismatch.
orders_wrong_type_df = orders_df.withColumn(
    'net_sales',
    F.col('net_sales').cast('string'),
)

try:
    validate_schema(
        orders_wrong_type_df,
        expected_schema=ORDERS_SCHEMA,
        label='orders_wrong_type_df',
    )
except ValueError as error:
    print(error)


<a id="experiment-2"></a>
# Experiment 2 — Required Fields

Declared order grain:

```text
one row per order_id
```

Required-field rules are **row-level** rules.

For required string identifiers, both are invalid:

```text
NULL
''
```

Expected rejection reason for the provided data:

```text
MISSING_ORDER_ID
```

[Back to Table of Contents](#toc)


In [ ]:
def missing_string(column_name: str) -> Column:
    '''Return an invalidity condition for a required string field.'''

    # Blank strings are structurally present but still unusable as keys.
    return (
        F.col(column_name).isNull()
        | (F.trim(F.col(column_name)) == '')
    )


required_field_demo_df = (
    orders_df
    .withColumn(
        'missing_order_id',
        missing_string('order_id'),
    )
    .withColumn(
        'missing_customer_id',
        missing_string('customer_id'),
    )
    .withColumn(
        'missing_order_date',
        F.col('order_date').isNull(),
    )
    .withColumn(
        'missing_order_status',
        missing_string('order_status'),
    )
    .withColumn(
        'missing_net_sales',
        F.col('net_sales').isNull(),
    )
)

required_field_demo_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


<a id="experiment-3"></a>
# Experiment 3 — Domain, Range, and Business Rules

Three different row-level rule types:

```text
DOMAIN
order_status ∈ {COMPLETED, CANCELLED, PENDING}

RANGE
net_sales >= 0.00

CROSS-COLUMN BUSINESS RULE
if order_status = CANCELLED
then net_sales = 0.00
```

Expected failures:

```text
O003 → INVALID_ORDER_STATUS
O004 → NEGATIVE_NET_SALES
O008 → INVALID_CANCELLED_AMOUNT
```

[Back to Table of Contents](#toc)


In [ ]:
ALLOWED_ORDER_STATUSES = (
    'COMPLETED',
    'CANCELLED',
    'PENDING',
)

rule_demo_df = (
    orders_df
    .withColumn(
        'invalid_status',
        F.col('order_status').isNotNull()
        & ~F.col('order_status').isin(*ALLOWED_ORDER_STATUSES),
    )
    .withColumn(
        'negative_net_sales',
        F.col('net_sales').isNotNull()
        & (F.col('net_sales') < F.lit(Decimal('0.00'))),
    )
    .withColumn(
        'invalid_cancelled_amount',
        # This rule depends on the relationship between two columns.
        (F.col('order_status') == 'CANCELLED')
        & F.col('net_sales').isNotNull()
        & (F.col('net_sales') != F.lit(Decimal('0.00'))),
    )
)

rule_demo_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


In [ ]:
def add_rejection_reasons(
    df: DataFrame,
    rules: list[tuple[str, Column]],
) -> DataFrame:
    '''Attach every row-level rejection reason that applies.'''

    # Each rule contributes one reason only when its invalidity condition is true.
    reason_columns = [
        F.when(invalid_condition, F.lit(reason))
        for reason, invalid_condition in rules
    ]

    return df.withColumn(
        'rejection_reasons',
        # Remove NULL placeholders so valid rows receive an empty array.
        F.filter(
            F.array(*reason_columns),
            lambda reason: reason.isNotNull(),
        ),
    )


ORDER_ROW_RULES = [
    ('MISSING_ORDER_ID', missing_string('order_id')),
    ('MISSING_CUSTOMER_ID', missing_string('customer_id')),
    ('MISSING_ORDER_DATE', F.col('order_date').isNull()),
    ('MISSING_ORDER_STATUS', missing_string('order_status')),
    ('MISSING_NET_SALES', F.col('net_sales').isNull()),
    (
        'INVALID_ORDER_STATUS',
        F.col('order_status').isNotNull()
        & ~F.col('order_status').isin(*ALLOWED_ORDER_STATUSES),
    ),
    (
        'NEGATIVE_NET_SALES',
        F.col('net_sales').isNotNull()
        & (F.col('net_sales') < F.lit(Decimal('0.00'))),
    ),
    (
        'INVALID_CANCELLED_AMOUNT',
        (F.col('order_status') == 'CANCELLED')
        & F.col('net_sales').isNotNull()
        & (F.col('net_sales') != F.lit(Decimal('0.00'))),
    ),
]

orders_row_validated_df = add_rejection_reasons(
    orders_df,
    rules=ORDER_ROW_RULES,
)

orders_row_validated_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


<a id="experiment-4"></a>
# Experiment 4 — Primary-Key Uniqueness

Declared grain:

```text
orders_df
= one row per order_id
```

Therefore:

```text
order_id must be present
AND
order_id must be unique
```

Primary-key uniqueness is a **dataset-level** rule.

Expected duplicate keys:

```text
O006
O007
```

[Back to Table of Contents](#toc)


In [ ]:
def find_duplicate_keys(
    df: DataFrame,
    key_columns: list[str],
) -> DataFrame:
    '''Return one row per duplicated business key.'''

    # groupBy() redistributes equal keys together, so this requires a shuffle.
    return (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col('count') > 1)
        .select(*key_columns)
    )


duplicate_order_ids_df = (
    find_duplicate_keys(
        orders_df,
        key_columns=['order_id'],
    )
    # NULL primary keys are handled by the required-field rule instead.
    .filter(F.col('order_id').isNotNull())
)

duplicate_order_ids_df.orderBy('order_id').show(truncate=False)

duplicate_order_rows_df = orders_df.join(
    duplicate_order_ids_df,
    on='order_id',
    how='left_semi',
)

duplicate_order_rows_df.orderBy('order_id', 'net_sales').show(truncate=False)


<a id="experiment-5"></a>
# Experiment 5 — Composite-Key Validation

Declared inventory grain:

```text
one row per snapshot_date + store_id + product_id
```

The individual columns may repeat.

The **combination** must be unique.

Expected duplicate composite key:

```text
2026-09-05 + S002 + P001
```

[Back to Table of Contents](#toc)


In [ ]:
inventory_key = [
    'snapshot_date',
    'store_id',
    'product_id',
]

duplicate_inventory_keys_df = find_duplicate_keys(
    inventory_df,
    key_columns=inventory_key,
)

duplicate_inventory_keys_df.show(truncate=False)

duplicate_inventory_rows_df = inventory_df.join(
    duplicate_inventory_keys_df,
    on=inventory_key,
    how='left_semi',
)

duplicate_inventory_rows_df.orderBy(
    'snapshot_date',
    'store_id',
    'product_id',
    'quantity_on_hand',
).show(truncate=False)


<a id="experiment-6"></a>
# Experiment 6 — Exact Duplicates vs. Duplicate Business Keys

These are different concepts.

```text
O006
→ duplicate business key with conflicting values

O007
→ duplicate business key AND exact duplicate row
```

`dropDuplicates()` should not be used as a generic validation strategy because it hides the defect and may choose a survivor without a defined business rule.

[Back to Table of Contents](#toc)


In [ ]:
def find_exact_duplicate_rows(df: DataFrame) -> DataFrame:
    '''Return one row per exact duplicated record value.'''

    # Group on every business column to distinguish exact duplicate records
    # from business-key collisions with differing non-key attributes.
    return (
        df
        .groupBy(*df.columns)
        .count()
        .filter(F.col('count') > 1)
    )


print('DUPLICATE BUSINESS KEYS')
find_duplicate_keys(
    orders_df,
    key_columns=['order_id'],
).orderBy('order_id').show(truncate=False)

print('EXACT DUPLICATE ROWS')
find_exact_duplicate_rows(
    orders_df
).orderBy('order_id').show(truncate=False)


<a id="experiment-7"></a>
# Experiment 7 — Referential Integrity

Relationship:

```text
orders.customer_id
    →
customers.customer_id
```

Referential integrity is a **cross-dataset** rule.

Expected orphan:

```text
O005 → customer_id C999
```

Before trusting the relationship, verify that the parent relation itself has the expected grain:

```text
customers_df
= one row per customer_id
```

[Back to Table of Contents](#toc)


In [ ]:
duplicate_customer_ids_df = find_duplicate_keys(
    customers_df,
    key_columns=['customer_id'],
)

duplicate_customer_ids_df.show(truncate=False)

# left_anti returns child rows for which no parent key exists.
orphan_orders_df = orders_df.join(
    customers_df.select('customer_id'),
    on='customer_id',
    how='left_anti',
)

orphan_orders_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


<a id="experiment-8"></a>
# Experiment 8 — Multiple Rejection Reasons

A row may violate more than one rule.

For diagnostics, preserve all meaningful failures rather than collapsing them into:

```text
INVALID_ROW
```

We'll add one synthetic row:

```text
order_id = NULL
customer_id = C999
order_status = UNKNOWN
net_sales = -10.00
```

Expected reasons include:

```text
MISSING_ORDER_ID
INVALID_ORDER_STATUS
NEGATIVE_NET_SALES
ORPHAN_CUSTOMER_ID
```

[Back to Table of Contents](#toc)


In [ ]:
MULTI_FAILURE_ROWS = [
    (
        None,
        'C999',
        date(2026, 9, 6),
        'UNKNOWN',
        Decimal('-10.00'),
    ),
]

multi_failure_df = spark.createDataFrame(
    MULTI_FAILURE_ROWS,
    schema=ORDERS_SCHEMA,
)

multi_failure_row_validated_df = add_rejection_reasons(
    multi_failure_df,
    rules=ORDER_ROW_RULES,
)

multi_failure_row_validated_df.show(truncate=False)


In [ ]:
def append_reason(
    df: DataFrame,
    reason: str,
    invalid_condition: Column,
) -> DataFrame:
    '''Append one reason while preserving reasons already attached.'''

    return df.withColumn(
        'rejection_reasons',
        F.when(
            invalid_condition,
            F.array_union(
                F.col('rejection_reasons'),
                F.array(F.lit(reason)),
            ),
        ).otherwise(F.col('rejection_reasons')),
    )


def add_duplicate_order_reason(
    validated_orders_df: DataFrame,
) -> DataFrame:
    '''Mark every non-null duplicated order_id.'''

    duplicate_keys_df = (
        find_duplicate_keys(
            validated_orders_df,
            key_columns=['order_id'],
        )
        .filter(F.col('order_id').isNotNull())
        .withColumn('_duplicate_order_id', F.lit(True))
    )

    result_df = (
        validated_orders_df
        .join(
            duplicate_keys_df,
            on='order_id',
            how='left',
        )
        .withColumn(
            '_duplicate_order_id',
            F.coalesce(
                F.col('_duplicate_order_id'),
                F.lit(False),
            ),
        )
    )

    result_df = append_reason(
        result_df,
        reason='DUPLICATE_ORDER_ID',
        invalid_condition=F.col('_duplicate_order_id'),
    )

    return result_df.drop('_duplicate_order_id')


def add_orphan_customer_reason(
    validated_orders_df: DataFrame,
    customers_df: DataFrame,
) -> DataFrame:
    '''Mark child rows whose non-null customer_id has no parent.'''

    duplicate_parent_exists = (
        find_duplicate_keys(
            customers_df,
            key_columns=['customer_id'],
        )
        .limit(1)
        .count()
        > 0
    )

    if duplicate_parent_exists:
        raise ValueError(
            'customers_df violates expected grain: customer_id is not unique.'
        )

    parent_keys_df = (
        customers_df
        .select('customer_id')
        .filter(F.col('customer_id').isNotNull())
        .withColumn('_customer_exists', F.lit(True))
    )

    result_df = (
        validated_orders_df
        .join(
            parent_keys_df,
            on='customer_id',
            how='left',
        )
        .withColumn(
            '_customer_exists',
            F.coalesce(
                F.col('_customer_exists'),
                F.lit(False),
            ),
        )
    )

    orphan_condition = (
        F.col('customer_id').isNotNull()
        & ~F.col('_customer_exists')
    )

    result_df = append_reason(
        result_df,
        reason='ORPHAN_CUSTOMER_ID',
        invalid_condition=orphan_condition,
    )

    return result_df.drop('_customer_exists')


def validate_orders(
    orders_df: DataFrame,
    customers_df: DataFrame,
) -> DataFrame:
    '''Apply structural, row-level, key, and relationship validation.'''

    validate_schema(
        orders_df,
        expected_schema=ORDERS_SCHEMA,
        label='orders_df',
    )
    validate_schema(
        customers_df,
        expected_schema=CUSTOMERS_SCHEMA,
        label='customers_df',
    )

    validated_df = add_rejection_reasons(
        orders_df,
        rules=ORDER_ROW_RULES,
    )
    validated_df = add_duplicate_order_reason(validated_df)
    validated_df = add_orphan_customer_reason(
        validated_df,
        customers_df,
    )

    return validated_df


In [ ]:
# Prove that one row can carry multiple independent failure reasons.
multi_failure_validated_df = validate_orders(
    multi_failure_df,
    customers_df,
)

multi_failure_validated_df.show(truncate=False)


<a id="experiment-9"></a>
# Experiment 9 — Accepted vs. Rejected Records

Split only **after** all required validation rules have been evaluated.

Classification invariant:

```text
Every interpretable input row
→ accepted OR rejected
```

Accepted rows:

```text
size(rejection_reasons) == 0
```

Rejected rows:

```text
size(rejection_reasons) > 0
```

[Back to Table of Contents](#toc)


In [ ]:
def split_accepted_rejected(
    validated_df: DataFrame,
) -> tuple[DataFrame, DataFrame]:
    '''Split rows after all required rejection reasons are attached.'''

    accepted_df = (
        validated_df
        .filter(F.size('rejection_reasons') == 0)
        .drop('rejection_reasons')
    )

    rejected_df = validated_df.filter(
        F.size('rejection_reasons') > 0
    )

    return accepted_df, rejected_df


validated_orders_df = validate_orders(
    orders_df,
    customers_df,
)

accepted_orders_df, rejected_orders_df = split_accepted_rejected(
    validated_orders_df,
)

print('VALIDATED ORDERS')
validated_orders_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)

print('ACCEPTED ORDERS')
accepted_orders_df.orderBy('order_id').show(truncate=False)

print('REJECTED ORDERS')
rejected_orders_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


<a id="experiment-10"></a>
# Experiment 10 — Quarantine Design

Rejected records should become a first-class output.

Preserve:

```text
original business columns
rejection_reasons
```

Add deterministic metadata:

```text
source_dataset
validation_run_date
```

Do not mutate away the original source defect before quarantine.

[Back to Table of Contents](#toc)


In [ ]:
def build_quarantine(
    rejected_df: DataFrame,
    source_dataset: str,
    validation_run_date: date,
) -> DataFrame:
    '''Add deterministic quarantine metadata without changing source evidence.'''

    return (
        rejected_df
        .withColumn(
            'source_dataset',
            F.lit(source_dataset),
        )
        # Inject the run date instead of depending on wall-clock execution time.
        .withColumn(
            'validation_run_date',
            F.lit(validation_run_date).cast('date'),
        )
    )


orders_quarantine_df = build_quarantine(
    rejected_orders_df,
    source_dataset='orders',
    validation_run_date=date(2026, 9, 7),
)

orders_quarantine_df.orderBy(
    F.col('order_id').asc_nulls_first()
).show(truncate=False)


<a id="experiment-11"></a>
# Experiment 11 — Validation Metrics and Reconciliation

Core metrics:

```text
input_count
accepted_count
rejected_count
acceptance_rate
rejection_rate
rejection count by reason
```

Mandatory reconciliation:

```text
input_count
=
accepted_count + rejected_count
```

Because a rejected row can fail multiple rules:

```text
sum(reason counts)
can exceed
rejected_count
```

[Back to Table of Contents](#toc)


In [ ]:
def build_validation_summary(
    validated_df: DataFrame,
    dataset_name: str,
) -> DataFrame:
    '''Build one summary row for the validation decision.'''

    return (
        validated_df
        .agg(
            F.count(F.lit(1)).alias('input_count'),
            F.sum(
                F.when(
                    F.size('rejection_reasons') == 0,
                    F.lit(1),
                ).otherwise(F.lit(0))
            ).alias('accepted_count'),
            F.sum(
                F.when(
                    F.size('rejection_reasons') > 0,
                    F.lit(1),
                ).otherwise(F.lit(0))
            ).alias('rejected_count'),
        )
        .withColumn(
            'dataset_name',
            F.lit(dataset_name),
        )
        .withColumn(
            'acceptance_rate',
            F.col('accepted_count') / F.col('input_count'),
        )
        .withColumn(
            'rejection_rate',
            F.col('rejected_count') / F.col('input_count'),
        )
        .select(
            'dataset_name',
            'input_count',
            'accepted_count',
            'rejected_count',
            'acceptance_rate',
            'rejection_rate',
        )
    )


def build_rejection_reason_counts(
    rejected_df: DataFrame,
) -> DataFrame:
    '''Count failures by machine-readable rejection reason.'''

    return (
        rejected_df
        .select(
            F.explode('rejection_reasons').alias('rejection_reason')
        )
        .groupBy('rejection_reason')
        .count()
    )


validation_summary_df = build_validation_summary(
    validated_orders_df,
    dataset_name='orders',
)

rejection_reason_counts_df = build_rejection_reason_counts(
    rejected_orders_df,
)

validation_summary_df.show(truncate=False)

rejection_reason_counts_df.orderBy(
    F.col('count').desc(),
    'rejection_reason',
).show(truncate=False)


In [ ]:
# Reconciliation is a correctness check, so materializing one summary row
# here is deliberate rather than accidental logging work.
summary = validation_summary_df.first()

assert summary['input_count'] == (
    summary['accepted_count'] + summary['rejected_count']
)

print('Validation reconciliation passed.')


<a id="experiment-12"></a>
# Experiment 12 — Distributed Cost of Validation

Predict before inspecting:

```text
row-level expressions
→ usually narrow

groupBy() uniqueness checks
→ shuffle / Exchange

referential-integrity joins
→ distributed join strategy

count / first / show / write
→ actions or materialization
```

Correctness comes first. Expensive validation should be optimized with evidence, not deleted because it costs Spark work.

[Back to Table of Contents](#toc)


In [ ]:
print('PRIMARY-KEY UNIQUENESS PLAN')

(
    orders_df
    .groupBy('order_id')
    .count()
    .filter(F.col('count') > 1)
    .explain('formatted')
)


In [ ]:
print('REFERENTIAL-INTEGRITY PLAN')

(
    orders_df
    .join(
        customers_df.select('customer_id'),
        on='customer_id',
        how='left_anti',
    )
    .explain('formatted')
)


<a id="applied-project"></a>
# Applied Phase 10 Project

Build one reusable retail data-quality layer spanning:

```text
orders
customers
inventory
```

Required architecture:

```text
Raw
 ↓
Schema enforcement
 ↓
Reusable validation
 ├── Accepted → transformations
 └── Rejected → quarantine
```

The important design rule is:

```text
validation owns source correctness
transformations own business results
```

[Back to Table of Contents](#toc)


In [ ]:
def validate_inventory(
    inventory_df: DataFrame,
) -> DataFrame:
    '''Apply row-level and composite-key inventory validation.'''

    validate_schema(
        inventory_df,
        expected_schema=INVENTORY_SCHEMA,
        label='inventory_df',
    )

    inventory_rules = [
        (
            'MISSING_SNAPSHOT_DATE',
            F.col('snapshot_date').isNull(),
        ),
        (
            'MISSING_STORE_ID',
            missing_string('store_id'),
        ),
        (
            'MISSING_PRODUCT_ID',
            missing_string('product_id'),
        ),
        (
            'MISSING_QUANTITY_ON_HAND',
            F.col('quantity_on_hand').isNull(),
        ),
        (
            'NEGATIVE_QUANTITY_ON_HAND',
            F.col('quantity_on_hand').isNotNull()
            & (F.col('quantity_on_hand') < 0),
        ),
    ]

    validated_df = add_rejection_reasons(
        inventory_df,
        rules=inventory_rules,
    )

    inventory_key = [
        'snapshot_date',
        'store_id',
        'product_id',
    ]

    duplicate_keys_df = (
        find_duplicate_keys(
            validated_df,
            key_columns=inventory_key,
        )
        .withColumn(
            '_duplicate_inventory_key',
            F.lit(True),
        )
    )

    validated_df = (
        validated_df
        .join(
            duplicate_keys_df,
            on=inventory_key,
            how='left',
        )
        .withColumn(
            '_duplicate_inventory_key',
            F.coalesce(
                F.col('_duplicate_inventory_key'),
                F.lit(False),
            ),
        )
    )

    validated_df = append_reason(
        validated_df,
        reason='DUPLICATE_INVENTORY_KEY',
        invalid_condition=F.col('_duplicate_inventory_key'),
    )

    return validated_df.drop('_duplicate_inventory_key')


validated_inventory_df = validate_inventory(inventory_df)

accepted_inventory_df, rejected_inventory_df = split_accepted_rejected(
    validated_inventory_df,
)

print('VALIDATED INVENTORY')
validated_inventory_df.orderBy(
    'snapshot_date',
    'store_id',
    'product_id',
).show(truncate=False)

print('ACCEPTED INVENTORY')
accepted_inventory_df.orderBy(
    'snapshot_date',
    'store_id',
    'product_id',
).show(truncate=False)

print('REJECTED INVENTORY')
rejected_inventory_df.orderBy(
    'snapshot_date',
    'store_id',
    'product_id',
).show(truncate=False)


In [ ]:
def transform_accepted_orders(
    accepted_orders_df: DataFrame,
    customers_df: DataFrame,
) -> DataFrame:
    '''Example business transformation that receives accepted data only.'''

    # Source-quality filters do not belong here because validation already owns
    # that responsibility.
    customer_projection_df = customers_df.select(
        'customer_id',
        'province',
        'customer_segment',
    )

    return accepted_orders_df.join(
        customer_projection_df,
        on='customer_id',
        how='left',
    )


enriched_accepted_orders_df = transform_accepted_orders(
    accepted_orders_df,
    customers_df,
)

enriched_accepted_orders_df.orderBy('order_id').show(truncate=False)


## Applied Review

The completed notebook now demonstrates:

```text
schema validation
required fields
domain validation
range checks
multi-column business rules
primary-key uniqueness
composite-key uniqueness
exact duplicate detection
duplicate business-key detection
referential integrity
multiple rejection reasons
accepted vs. rejected records
quarantine metadata
validation metrics
reconciliation
distributed execution review
validation / transformation separation
```

Key architectural outcome:

```text
raw_orders_df
    ↓
validate_orders(...)
    ↓
validated_orders_df
    ├── accepted_orders_df
    │       ↓
    │   business transformations
    │
    └── rejected_orders_df
            ↓
        quarantine
```

This notebook provides worked practice. It still does **not** satisfy the formal Phase 10 mastery gate by itself.

[Back to Table of Contents](#toc)


<a id="cleanup"></a>
# Cleanup

Stop the local teaching session when finished.

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()
